<a href="https://colab.research.google.com/github/raimund-buehler/visualizing-the-brain/blob/update-day-one-notebook/notebooks/day_one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day One: Meet MRI and fMRI Data

Today we will use Python to open brain images, inspect the numbers inside them, and explore the brain slice by slice.

By the end, you will be able to:

- explain the difference between an anatomical MRI image and an fMRI time series,
- describe pixels, voxels, slices, volumes, and 4D data,
- inspect the shape and values of a brain image,
- make several brain plots with Nilearn, and
- plot the fMRI signal from one voxel over time.

## What is a Jupyter notebook?

A Jupyter notebook is a page where text, code, pictures, and results can live together.

This notebook has two kinds of boxes, called **cells**:

- **Text cells**, like this one, explain what is happening.
- **Code cells** contain Python instructions that the computer can run.

To run a code cell, click the play button on the left. You can also click inside the cell and press **Shift + Enter**.

Run the cells from top to bottom. A later cell may need a variable that was created in an earlier cell.

## Setup: press play here first

The next cell installs and imports our tools. It also loads a standard anatomical brain template.

You do not need to understand every setup line yet. Run it once and wait for **Setup complete**.

In [ ]:
# @title Setup: install tools and load an anatomical template
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "nilearn", "nibabel", "matplotlib", "numpy"
    ])

import numpy as np
import matplotlib.pyplot as plt
from nilearn import datasets, image, plotting

%matplotlib inline

# This is an average anatomical brain in a shared reference space.
anat_img = datasets.load_mni152_template(resolution=2)

print("Setup complete.")
print("Anatomical image shape:", anat_img.shape)

## MRI data: anatomy and function

MRI scanners can create different kinds of data. We will use two today:

- An **anatomical MRI** is a detailed 3D picture of brain structure. Each voxel has one brightness value.
- **fMRI** records many 3D brain volumes, one after another. Each voxel therefore has a series of values over time.

A scanner collects the brain in **slices**. A stack of slices makes one 3D **volume**. In fMRI, many volumes make a 4D data set.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/mri_fmri_build_up.png" alt="MRI and fMRI data built from voxels, slices, volumes, and time" width="850">

The highlighted green cube is one **voxel**. A voxel is like a 3D pixel.

## A brain image is an array of numbers

A computer does not begin with a picture of a brain. It stores a grid of numbers called an **array**. Plotting software turns those numbers into brightness or color.

For a 3D anatomical image, the array has three dimensions:

- the first position moves through the **x** direction,
- the second moves through the **y** direction, and
- the third moves through the **z** direction.

The image's `shape` tells us how many voxel positions exist along each dimension.

In [ ]:
# Turn the image data into a NumPy array that we can inspect.
anat_data = anat_img.get_fdata()

print("Python object:", type(anat_img))
print("Array type:", type(anat_data))
print("Array shape:", anat_data.shape)
print("Number of dimensions:", anat_data.ndim)

### Reading one voxel

We can use three array positions to ask for one voxel value. Python starts counting at **0**, not 1.

The code below finds the middle array position automatically. The three numbers are an **array index**. They identify a stored voxel, but they are not yet millimetre coordinates on a brain map.

In [ ]:
# Find the middle index along x, y, and z.
middle_x = anat_data.shape[0] // 2
middle_y = anat_data.shape[1] // 2
middle_z = anat_data.shape[2] // 2

middle_value = anat_data[middle_x, middle_y, middle_z]

print("Middle array index:", (middle_x, middle_y, middle_z))
print("Value stored there:", middle_value)

An array can also give us a small block of nearby voxels. The colon in `a:b` means: start at `a` and stop just before `b`.

In [ ]:
# Select a 3 by 3 by 3 block around the middle.
small_block = anat_data[
    middle_x - 1:middle_x + 2,
    middle_y - 1:middle_y + 2,
    middle_z - 1:middle_z + 2,
]

print("Block shape:", small_block.shape)
print(small_block)

## Brain slices and directions

Scientists view the 3D array as slices from three directions:

- **Sagittal** slices divide left and right. Nilearn calls this the `x` direction.
- **Coronal** slices divide front and back. Nilearn calls this the `y` direction.
- **Axial** slices divide top and bottom. Nilearn calls this the `z` direction.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/AnatomicalPlanes.png" alt="Sagittal, coronal, and axial brain planes" width="700">

## Plot 1: three directions at once

`plot_anat` is designed for anatomical MRI images. The default `ortho` view shows one sagittal, one coronal, and one axial slice together. The crosshairs mark the same location in all three views.

In [ ]:
plotting.plot_anat(
    anat_img,
    display_mode="ortho",
    title="Anatomical MRI: three slice directions",
)
plotting.show()

## Plot 2: a row of axial slices

Sometimes we want to compare several slices from the same direction. Here, `display_mode="z"` asks for axial slices, and `cut_coords` chooses their heights in millimetres.

These map coordinates are different from the array indices we used earlier. Nilearn reads the image information needed to place the voxels in brain space.

In [ ]:
slice_heights = [-30, -10, 10, 30, 50, 70]

plotting.plot_anat(
    anat_img,
    display_mode="z",
    cut_coords=slice_heights,
    title="Axial slices from lower to higher positions",
)
plotting.show()

## Plot 3: an interactive viewer

`view_img` creates a viewer you can click. It is useful for exploring many locations instead of making one fixed figure.

Try clicking in each view. Watch how the crosshair and the `x`, `y`, and `z` coordinates change.

In [ ]:
anat_viewer = plotting.view_img(
    anat_img,
    bg_img=False,
    cmap="gray",
    colorbar=False,
    symmetric_cmap=False,
    title="Click to explore the anatomical template",
)

anat_viewer

### Quick challenge

1. In the axial-slice code, change one number in `slice_heights` and run the cell again.
2. Change `display_mode="z"` to `display_mode="x"`. Which slice direction appears?
3. Use the interactive viewer to find a location where the left and right sides look nearly symmetrical.

## fMRI adds a fourth dimension: time

An anatomical image has a value at every `(x, y, z)` voxel position. An fMRI image adds a fourth position, `t`, for **time**.

We can think about the same 4D data in two ways:

1. a sequence of 3D brain volumes, or
2. a time series from every voxel.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/fmri_4d_timeseries.png" alt="A sequence of fMRI volumes and a time series from one voxel" width="700">

The **repetition time**, often shortened to **TR**, is the time between one whole-brain volume and the next.

### What is the fMRI signal?

The scanner stores a signal-intensity number for each voxel at each time point. That number is influenced by the **BOLD signal**: changes in blood oxygen that are connected to brain activity.

The line from one voxel can rise and fall for several reasons, including:

- changes linked to a task and local brain activity,
- breathing, heartbeat, and head movement,
- scanner noise, and
- slow changes during the scan.

So a bump in one raw time series is **not automatically brain activation**. Later analysis compares the measured signal with what happened during the experiment.

## Load a real example fMRI scan

Nilearn provides a small teaching data set from one participant. The participant performed simple tasks such as reading, listening, looking at checkerboards, and pressing buttons.

The next cell downloads the scan the first time it is run. This may take a minute.

In [ ]:
# Download one preprocessed 4D fMRI image.
fmri_dataset = datasets.fetch_localizer_first_level()
fmri_img = image.load_img(fmri_dataset.epi_img)

print("fMRI image shape:", fmri_img.shape)
print("Number of dimensions:", len(fmri_img.shape))

The first three shape numbers count voxel positions in space. The last number counts time points.

For example, a shape of `(53, 63, 46, 128)` would mean 53 by 63 by 46 spatial positions, measured at 128 time points. Read the actual numbers printed above.

## Plot 4: one volume and the mean fMRI image

A plotting function needs a 3D image, so we can select one time point with `index_img`. We can also use `mean_img` to average all time points into one 3D image.

`plot_epi` is useful for functional MRI volumes. fMRI images usually look blurrier and less detailed than anatomical MRI images.

In [ ]:
first_volume = image.index_img(fmri_img, 0)

plotting.plot_epi(
    first_volume,
    title="The first 3D volume in the fMRI scan",
)
plotting.show()

In [ ]:
mean_fmri = image.mean_img(fmri_img, copy_header=True)

plotting.plot_epi(
    mean_fmri,
    title="Mean across all fMRI time points",
)
plotting.show()

## Extract one voxel's time series

To select one whole 3D volume, we keep every `x`, `y`, and `z` position and choose one time point:

```python
fmri_data[:, :, :, time_point]
```

To select one voxel's full time series, we choose one `x`, `y`, and `z` position and keep every time point:

```python
fmri_data[x, y, z, :]
```

In [ ]:
# Choose the middle array position in the fMRI image.
voxel_x = fmri_img.shape[0] // 2
voxel_y = fmri_img.shape[1] // 2
voxel_z = fmri_img.shape[2] // 2

# Read this voxel at every time point.
voxel_signal = np.asarray(
    fmri_img.dataobj[voxel_x, voxel_y, voxel_z, :]
)

print("Voxel index:", (voxel_x, voxel_y, voxel_z))
print("Time-series shape:", voxel_signal.shape)
print("First 10 signal values:", voxel_signal[:10])

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(voxel_signal, color="royalblue")
plt.xlabel("Time point (volume number)")
plt.ylabel("fMRI signal intensity")
plt.title("Signal over time from one voxel")
plt.grid(alpha=0.25)
plt.show()

### Time-series challenge

Change `voxel_x`, `voxel_y`, or `voxel_z` by a small amount, then rerun the extraction and plotting cells.

- Does the line have the same shape?
- Does its average height change?
- What might explain differences between nearby voxels?

Be careful not to choose a number below 0 or above the image shape.

## Which plotting tool should I use?

| Tool | Good for | Output |
|---|---|---|
| `plot_anat` | Detailed anatomical MRI images | A fixed figure |
| `plot_epi` | Functional MRI volumes or their mean | A fixed figure |
| `view_img` | Clicking through many brain locations | An interactive viewer |

A fixed plot is useful for a report or presentation. An interactive viewer is useful while exploring and checking data.

## Day-one recap

- MRI images are arrays of numbers.
- A voxel is a 3D pixel, slices make a volume, and many fMRI volumes make 4D data.
- A 3D anatomical image has `(x, y, z)` dimensions.
- A 4D fMRI image has `(x, y, z, time)` dimensions.
- Each voxel in fMRI has a signal time series. BOLD contributes to that signal, but noise and movement do too.
- Nilearn can create fixed slice plots and interactive viewers.

On day two, we can connect the changing fMRI signal to the timing of an experiment and create an activity map.

## Sources and further reading

This notebook adapts the MRI-data structure and 4D time-series teaching figures from `TEWA2/02_Understanding_MRI_Data.ipynb` in this repository.

- [Nilearn: 3D and 4D images](https://nilearn.github.io/stable/auto_examples/00_tutorials/plot_3d_and_4d_niimg.html)
- [Nilearn plotting tools](https://nilearn.github.io/stable/modules/plotting.html)
- Pinel et al. (2007), the example localizer fMRI data set: https://doi.org/10.1186/1471-2202-8-91